SPLIT THEM TO 3 FOLDERS

In [ ]:
from pathlib import Path
import shutil, json, csv, random, re
from collections import Counter, defaultdict
from datetime import datetime

# =========================
# CONFIG
# =========================
STAGE1_INTERIM = Path("../../data/interim/Stage1/gray_clahe_1500x1000_noborder_aug")
STAGE2_INTERIM = Path("../../data/interim/Stage2/gray_clahe_1500x1000_noborder_aug")

OUT_PROCESSED = Path("../../data/processed/Stage2/gray/combined")
OUT_IMAGES = OUT_PROCESSED / "images"
OUT_INDEX  = OUT_PROCESSED / "index.csv"
OUT_SPLITS = OUT_PROCESSED / "splits.json"

OUT_LABELS_DIR = Path("../../data/labels/Stage2/gray")
OUT_LABELS_DIR.mkdir(parents=True, exist_ok=True)
OUT_TRAIN_LBL = OUT_LABELS_DIR / "train.json"
OUT_VAL_LBL   = OUT_LABELS_DIR / "val.json"
OUT_TEST_LBL  = OUT_LABELS_DIR / "test.json"

TRAIN_RATIO = 0.85
VAL_RATIO   = 0.10
TEST_RATIO  = 0.05
SEED = 42

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}

# stage -> label
STAGE_TO_OVERLAP = {
    "stage1": 0,
    "stage2": 1,
}

# =========================
# HELPERS
# =========================
def extract_timestamp(filename: str) -> str:
    m = re.search(r"(\d{4}-\d{2}-\d{2})_(\d{2}-\d{2}-\d{2})-(\d{3})", filename)
    if not m:
        return ""
    return f"{m.group(1)}T{m.group(2).replace('-', ':')}.{m.group(3)}"

def list_images(root: Path):
    return sorted([p for p in root.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

def base_group_id(filename: str) -> str:
    """
    Group orig + augXX together so they NEVER split apart.
    Example:
      2026-01-21_15-51-50-237_orig.png
      2026-01-21_15-51-50-237_aug01.png
    -> group id = 2026-01-21_15-51-50-237
    """
    # remove suffix like _orig or _augXX before extension
    # keep the timestamp prefix
    m = re.match(r"(.+?)_(orig|aug\d+)\.[^.]+$", filename)
    if m:
        return m.group(1)
    # if no match, group by filename stem
    return Path(filename).stem

def copy_stage_grouped(stage_root: Path, stage_key: str, seed: int):
    """
    Returns: dict split->list[Path], plus per-file metadata rows
    Splits by GROUPS (base_group_id) to avoid leakage.
    """
    all_paths = list_images(stage_root)
    assert all_paths, f"No images found in {stage_root}"

    groups = defaultdict(list)
    for p in all_paths:
        gid = base_group_id(p.name)
        groups[gid].append(p)

    group_ids = sorted(groups.keys())
    rng = random.Random(seed)
    rng.shuffle(group_ids)

    n_groups = len(group_ids)
    n_train = int(n_groups * TRAIN_RATIO)
    n_val   = int(n_groups * VAL_RATIO)

    train_g = set(group_ids[:n_train])
    val_g   = set(group_ids[n_train:n_train + n_val])
    test_g  = set(group_ids[n_train + n_val:])

    splits = {"train": [], "val": [], "test": []}
    rows = []

    for gid, plist in groups.items():
        if gid in train_g:
            split = "train"
        elif gid in val_g:
            split = "val"
        else:
            split = "test"

        for p in plist:
            splits[split].append(p)
            rows.append({
                "stage": stage_key,
                "split": split,
                "src_path": str(p),
                "filename": p.name,
                "group_id": gid,
                "scan_timestamp": extract_timestamp(p.name),
            })

    return splits, rows

def write_outputs(all_rows, out_root: Path):
    # Make folders
    for sp in ("train", "val", "test"):
        (OUT_IMAGES / sp).mkdir(parents=True, exist_ok=True)

    # Copy files + build final index rows
    final_rows = []
    splits_json = {"train": [], "val": [], "test": []}

    for r in all_rows:
        stage = r["stage"]
        split = r["split"]
        src = Path(r["src_path"])

        # prefix to prevent collisions between stage1/stage2 filenames
        new_name = f"{stage}__{src.name}"
        dst_rel = f"images/{split}/{new_name}"
        dst_abs = out_root / dst_rel

        shutil.copy2(src, dst_abs)

        final_rows.append({
            "filepath": dst_rel,
            "split": split,
            "stage": stage,
            "scan_session_id": src.parent.name,      # folder name
            "filename": new_name,
            "scan_timestamp": r["scan_timestamp"],
            "source_interim_path": str(src),
            "group_id": r["group_id"],
        })
        splits_json[split].append(dst_rel)

    # Save splits.json
    with open(OUT_SPLITS, "w") as f:
        json.dump({
            "created_at": datetime.now().isoformat(timespec="seconds"),
            "sources": {
                "stage1": str(STAGE1_INTERIM),
                "stage2": str(STAGE2_INTERIM),
            },
            "seed": SEED,
            "ratios": {"train": TRAIN_RATIO, "val": VAL_RATIO, "test": TEST_RATIO},
            "counts": {k: len(v) for k, v in splits_json.items()},
            "splits": splits_json,
        }, f, indent=2)

    # Save index.csv
    with open(OUT_INDEX, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(final_rows[0].keys()))
        w.writeheader()
        w.writerows(final_rows)

    return final_rows, splits_json

def make_labels_from_index(final_rows, split_name):
    items = []
    for r in final_rows:
        if r["split"] != split_name:
            continue
        st = r["stage"]
        items.append({
            "image": r["filepath"],              # matches index.csv exactly
            "overlap": STAGE_TO_OVERLAP[st],     # stage1=0, stage2=1
            "stage": st,
        })
    return items

# =========================
# RUN
# =========================
OUT_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Listing images...")
s1_splits, s1_rows = copy_stage_grouped(STAGE1_INTERIM, "stage1", seed=SEED + 101)
s2_splits, s2_rows = copy_stage_grouped(STAGE2_INTERIM, "stage2", seed=SEED + 202)

print("Stage1 counts:", {k: len(v) for k, v in s1_splits.items()})
print("Stage2 counts:", {k: len(v) for k, v in s2_splits.items()})

all_rows = s1_rows + s2_rows
final_rows, splits_json = write_outputs(all_rows, OUT_PROCESSED)

print("Saved combined dataset to:", OUT_PROCESSED)
print("Combined split counts:", {k: len(v) for k, v in splits_json.items()})
print("Stage dist:", Counter([r["stage"] for r in final_rows]))

# =========================
# LABELS (train/val/test)
# =========================
train_lbl = make_labels_from_index(final_rows, "train")
val_lbl   = make_labels_from_index(final_rows, "val")
test_lbl  = make_labels_from_index(final_rows, "test")

json.dump(train_lbl, open(OUT_TRAIN_LBL, "w"), indent=2)
json.dump(val_lbl,   open(OUT_VAL_LBL,   "w"), indent=2)
json.dump(test_lbl,  open(OUT_TEST_LBL,  "w"), indent=2)

print("Wrote labels:", OUT_TRAIN_LBL, OUT_VAL_LBL, OUT_TEST_LBL)
print("Train label dist:", Counter([x["overlap"] for x in train_lbl]))
print("Val label dist  :", Counter([x["overlap"] for x in val_lbl]))
print("Test label dist :", Counter([x["overlap"] for x in test_lbl]))

Automate Labelling for Stage1 since all data will have same labels

In [ ]:
# Split Stage2/combined TRAIN split into 3 folders: train1/, train2/, train3/
# leakage-safe: keeps siblings together (same base id)
# uses existing combined index.csv (so it matches your Stage2 combined dataset)
# writes:
#   - images/train1, images/train2, images/train3
#   - index_train123.csv
#   - splits_train123.json
#   - (optional) labels: train1.json, train2.json, train3.json (if train.json exists)

from pathlib import Path
import csv, json, random, re, shutil
from collections import defaultdict
from datetime import datetime

# =========================
# CONFIG
# =========================
PROCESSED_ROOT = Path("../../data/processed/Stage2/gray/combined")
INDEX_IN       = PROCESSED_ROOT / "index.csv"
IMAGES_DIR     = PROCESSED_ROOT / "images"

TRAIN_DIR      = IMAGES_DIR / "train"
OUT_TRAIN1     = IMAGES_DIR / "train1"
OUT_TRAIN2     = IMAGES_DIR / "train2"
OUT_TRAIN3     = IMAGES_DIR / "train3"

OUT_INDEX      = PROCESSED_ROOT / "index_train123.csv"
OUT_SPLITS     = PROCESSED_ROOT / "splits_train123.json"

# Optional: if you already have Stage2 labels/train.json (overlap labels)
LABELS_DIR     = Path("../../data/labels/Stage2/gray")
LABEL_TRAIN_IN = LABELS_DIR / "train.json"
LABEL_TRAIN1   = LABELS_DIR / "train1.json"
LABEL_TRAIN2   = LABELS_DIR / "train2.json"
LABEL_TRAIN3   = LABELS_DIR / "train3.json"

SEED = 42
MOVE_FILES = False   # False = copy train -> train1/2/3; True = move

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tiff"}

# =========================
# HELPERS
# =========================
def ensure_dir(d: Path):
    d.mkdir(parents=True, exist_ok=True)

def group_id_from_combined_filename(filename: str) -> str:
    """
    Combined filenames look like:
      stage1__2026-01-21_15-51-50-237_orig.png
      stage2__2026-01-21_15-51-50-237_aug01.png

    We want ONE group for all siblings:
      2026-01-21_15-51-50-237
    (and still keep stage prefix out of the group id)
    """
    name = filename

    # strip stage prefix if present
    name = re.sub(r"^stage\d+__", "", name, flags=re.IGNORECASE)

    # strip _orig / _augXX before extension
    name = re.sub(r"_(orig|aug\d+)(?=\.[^.]+$)", "", name, flags=re.IGNORECASE)

    return Path(name).stem  # stable base

def read_index_csv(p: Path):
    rows = []
    with open(p, "r", newline="") as f:
        for r in csv.DictReader(f):
            rr = dict(r)
            rr["filepath"] = (rr.get("filepath") or "").replace("\\", "/")
            rr["split"]    = (rr.get("split") or "").strip().lower()
            rows.append(rr)
    return rows

def write_index_csv(p: Path, rows: list, fieldnames: list):
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            w.writerow(r)

def place_files(subset_rows, out_dir: Path, split_name: str):
    ensure_dir(out_dir)
    relpaths = []
    missing = 0

    for r in subset_rows:
        fp = (r.get("filepath") or "").replace("\\","/").strip()
        if not fp:
            continue
        src = PROCESSED_ROOT / fp
        if not src.exists():
            missing += 1
            continue

        dst = out_dir / src.name
        if not dst.exists():
            if MOVE_FILES:
                shutil.move(str(src), str(dst))
            else:
                shutil.copy2(src, dst)

        relpaths.append(f"images/{split_name}/{dst.name}")

    return relpaths, missing

# =========================
# LOAD index + take TRAIN rows
# =========================
rows = read_index_csv(INDEX_IN)
if not rows:
    raise ValueError(f"Empty index: {INDEX_IN.resolve()}")

train_rows = [r for r in rows if r["split"] == "train"]
if not train_rows:
    raise ValueError("No split=train rows found in index.csv")

# =========================
# GROUP train rows (leakage-safe)
# =========================
groups = defaultdict(list)
skipped_missing = 0

for r in train_rows:
    fp = (r.get("filepath") or "").replace("\\","/").strip()
    if not fp:
        continue
    src = PROCESSED_ROOT / fp
    if not src.exists():
        skipped_missing += 1
        continue

    gid = group_id_from_combined_filename(Path(fp).name)
    groups[gid].append(r)

gids = list(groups.keys())
random.seed(SEED)
random.shuffle(gids)

n = len(gids)
cut1 = n // 3
cut2 = 2 * (n // 3)

gids1 = set(gids[:cut1])
gids2 = set(gids[cut1:cut2])
gids3 = set(gids[cut2:])

def rows_from_gidset(gset):
    out = []
    for gid in gset:
        out.extend(groups[gid])
    return out

train1_rows = rows_from_gidset(gids1)
train2_rows = rows_from_gidset(gids2)
train3_rows = rows_from_gidset(gids3)

print("Train groups:", n)
print("train1 groups/images:", len(gids1), len(train1_rows))
print("train2 groups/images:", len(gids2), len(train2_rows))
print("train3 groups/images:", len(gids3), len(train3_rows))
if skipped_missing:
    print("Missing train files on disk (skipped):", skipped_missing)

# =========================
# COPY/MOVE into train1/2/3 folders
# =========================
ensure_dir(OUT_TRAIN1)
ensure_dir(OUT_TRAIN2)
ensure_dir(OUT_TRAIN3)

train1_rel, miss1 = place_files(train1_rows, OUT_TRAIN1, "train1")
train2_rel, miss2 = place_files(train2_rows, OUT_TRAIN2, "train2")
train3_rel, miss3 = place_files(train3_rows, OUT_TRAIN3, "train3")

print("Copied/moved images:")
print(" train1:", len(train1_rel), "| missing:", miss1)
print(" train2:", len(train2_rel), "| missing:", miss2)
print(" train3:", len(train3_rel), "| missing:", miss3)

# =========================
# WRITE splits_train123.json
# =========================
meta = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source_index": str(INDEX_IN),
    "seed": SEED,
    "move_files": MOVE_FILES,
    "grouping": "strip stageX__ + strip _orig/_augXX",
    "splits": {"train1": train1_rel, "train2": train2_rel, "train3": train3_rel},
    "counts": {"train1": len(train1_rel), "train2": len(train2_rel), "train3": len(train3_rel)},
}
with open(OUT_SPLITS, "w") as f:
    json.dump(meta, f, indent=2)
print("Wrote:", OUT_SPLITS.resolve())

# =========================
# WRITE index_train123.csv (val/test unchanged)
# =========================
# map filename -> new split
split_by_fname = {}
for rel in train1_rel: split_by_fname[Path(rel).name] = "train1"
for rel in train2_rel: split_by_fname[Path(rel).name] = "train2"
for rel in train3_rel: split_by_fname[Path(rel).name] = "train3"

# keep original fieldnames if possible
fieldnames = list(rows[0].keys())
if "split" not in fieldnames: fieldnames.append("split")
if "filepath" not in fieldnames: fieldnames.append("filepath")

rewritten = 0
skipped = 0
new_rows = []

for r in rows:
    rr = dict(r)
    if rr["split"] == "train":
        old_fp = (rr.get("filepath") or "").replace("\\","/").strip()
        fname = Path(old_fp).name
        new_split = split_by_fname.get(fname)
        if new_split is None:
            skipped += 1
        else:
            rr["split"] = new_split
            rr["filepath"] = f"images/{new_split}/{fname}"
            rewritten += 1
    new_rows.append(rr)

write_index_csv(OUT_INDEX, new_rows, fieldnames)
print("Wrote:", OUT_INDEX.resolve())
print("Rewritten train rows:", rewritten)
print("Skipped train rows (not found in train1/2/3):", skipped)

# =========================
# OPTIONAL: split Stage2 labels/train.json -> train1/2/3 jsons
# (only if ../../data/labels/Stage2/train.json exists and uses full relpaths like images/train/xxx.png)
# =========================
if LABEL_TRAIN_IN.exists():
    train_lbl = json.load(open(LABEL_TRAIN_IN, "r"))
    # map by filename so we don't care if path had images/train/
    lbl_by_fname = {Path(it["image"].replace("\\","/")).name: it for it in train_lbl}

    def remap_labels(relpaths, out_path: Path, split_name: str):
        out = []
        for rel in relpaths:
            fname = Path(rel).name
            it = lbl_by_fname.get(fname)
            if it is None:
                continue
            it2 = dict(it)
            it2["image"] = f"images/{split_name}/{fname}"
            out.append(it2)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        json.dump(out, open(out_path, "w"), indent=2)
        print(f"Wrote labels: {out_path.resolve()} | items={len(out)}")

    remap_labels(train1_rel, LABEL_TRAIN1, "train1")
    remap_labels(train2_rel, LABEL_TRAIN2, "train2")
    remap_labels(train3_rel, LABEL_TRAIN3, "train3")
else:
    print("ℹ No Stage2 train labels found at:", LABEL_TRAIN_IN.resolve())
    print("   Skipped writing train1/train2/train3 label jsons.")

print("\nDONE  Train split is now also available as train1/train2/train3 folders.")
